In [ ]:
# =========================
# 1. Install dependencies
# =========================
# pip install pandas numpy scikit-learn torch transformers openai tqdm

# =========================
# 2. Imports + config
# =========================
import os
import json
import numpy as np
import pandas as pd
from tqdm.auto import tqdm

import torch
from torch.utils.data import Dataset

from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support,
    roc_auc_score,
    average_precision_score,
)

from transformers import (
    DistilBertTokenizerFast,
    DistilBertForSequenceClassification,
    Trainer,
    TrainingArguments,
)

from openai import OpenAI

DATA_PATH = "datset.csv"   # change to "dataset.csv" if needed
OPENAI_MODEL = "gpt-5-mini"   # lower-cost LLM for feature engineering
DISTILBERT_MODEL = "distilbert-base-uncased"
MAX_LEN = 256
LOOKBACK_TXNS = 10
USE_LLM = True
SEED = 42

np.random.seed(SEED)
torch.manual_seed(SEED)
import os


client = OpenAI()


/data1/rachit/.conda/envs/cond/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [5]:
DATA_PATH="../../transactions/card_transaction.v1.csv"
df = pd.read_csv(DATA_PATH)

In [ ]:
# 🔥 Model

from dotenv import load_dotenv
import os
import google.generativeai as genai

load_dotenv()

genai.configure(api_key=os.getenv("GEMINI_API_KEY"))

In [13]:
PROMPT = """
You are a fraud analytics feature engineering assistant.

Given a transaction dataset with columns:

['User', 'Card', 'Year', 'Month', 'Day', 'Time', 'Amount', 'Use Chip',
 'Merchant Name', 'Merchant City', 'Merchant State', 'Zip', 'MCC',
 'Errors?', 'Is Fraud?']

Your task is to design GLOBAL USER-LEVEL FEATURES.

These features must:
- be the SAME for all users
- be computable using pandas groupby on User
- use aggregation functions (mean, std, count, ratio, nunique, etc.)

Output STRICTLY in JSON:

[
  {
    "feature_name": "...",
    "pandas_code": "...",
    "description": "..."
  }
]

Generate 10–15 strong fraud detection features.

Do NOT compute values.
Do NOT use per-user custom logic.
Do NOT predict fraud.
"""

In [19]:
for m in genai.list_models():
    print(m.name, "->", m.supported_generation_methods)

models/gemini-2.5-flash -> ['generateContent', 'countTokens', 'createCachedContent', 'batchGenerateContent']
models/gemini-2.5-pro -> ['generateContent', 'countTokens', 'createCachedContent', 'batchGenerateContent']
models/gemini-2.0-flash -> ['generateContent', 'countTokens', 'createCachedContent', 'batchGenerateContent']
models/gemini-2.0-flash-001 -> ['generateContent', 'countTokens', 'createCachedContent', 'batchGenerateContent']
models/gemini-2.0-flash-lite-001 -> ['generateContent', 'countTokens', 'createCachedContent', 'batchGenerateContent']
models/gemini-2.0-flash-lite -> ['generateContent', 'countTokens', 'createCachedContent', 'batchGenerateContent']
models/gemini-2.5-flash-preview-tts -> ['countTokens', 'generateContent']
models/gemini-2.5-pro-preview-tts -> ['countTokens', 'generateContent', 'batchGenerateContent']
models/gemma-3-1b-it -> ['generateContent', 'countTokens']
models/gemma-3-4b-it -> ['generateContent', 'countTokens']
models/gemma-3-12b-it -> ['generateContent

In [25]:
model = genai.GenerativeModel("models/gemini-flash-lite-latest")


In [26]:
response = model.generate_content(PROMPT)

output = response.text
print("Raw Output:\n", output)

# 🧹 Clean JSON (Gemini sometimes adds ```json)
output = output.replace("```json", "").replace("```", "").strip()

features = json.loads(output)

# =========================================
# ✅ APPLY FEATURES USING PANDAS
# =========================================

# Example: load your dataset
# df = pd.read_csv("your_dataset.csv")

user_df = pd.DataFrame()

for f in features:
    try:
        print("Applying:", f["feature_name"])
        user_df[f["feature_name"]] = eval(f["pandas_code"])
    except Exception as e:
        print("Error in", f["feature_name"], ":", e)

# ✅ Add label
user_df["is_fraud_user"] = df.groupby("User")["Is Fraud?"].max()

# ✅ Fill missing
user_df = user_df.fillna(0)

print("\nFinal Shape:", user_df.shape)
print(user_df.head())

Raw Output:
 ```json
[
  {
    "feature_name": "User_Total_Transactions",
    "pandas_code": "df.groupby('User')['Amount'].count().rename('User_Total_Transactions')",
    "description": "The total number of transactions made by the user across the entire dataset."
  },
  {
    "feature_name": "User_Average_Amount",
    "pandas_code": "df.groupby('User')['Amount'].mean().rename('User_Average_Amount')",
    "description": "The mean transaction amount for the user across all recorded history."
  },
  {
    "feature_name": "User_Std_Amount",
    "pandas_code": "df.groupby('User')['Amount'].std().rename('User_Std_Amount')",
    "description": "The standard deviation of transaction amounts for the user, indicating spending volatility."
  },
  {
    "feature_name": "User_Unique_Merchants_Count",
    "pandas_code": "df.groupby('User')['Merchant Name'].nunique().rename('User_Unique_Merchants_Count')",
    "description": "The total number of unique merchants the user has transacted with."
  },
 

In [34]:
user_df["is_fraud_user"] = df.groupby("User")["Is Fraud?"].max()

In [35]:
# Separate label first
y = user_df["is_fraud_user"]

# Remove non-numeric features
X = user_df.drop(columns=["is_fraud_user"])
X = X.select_dtypes(include=["number"])

In [36]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y   # IMPORTANT for fraud datasets
)

In [37]:
from sklearn.ensemble import RandomForestClassifier

model = RandomForestClassifier(
    n_estimators=200,
    max_depth=10,
    random_state=42,
    class_weight="balanced"   # IMPORTANT for fraud
)

model.fit(X_train, y_train)

,n_estimators,200
,criterion,'gini'
,max_depth,10
,min_samples_split,2
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_features,'sqrt'
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,bootstrap,True
,oob_score,False


In [38]:
from sklearn.metrics import classification_report, roc_auc_score

y_pred = model.predict(X_test)
y_prob = model.predict_proba(X_test)[:, 1]

print(classification_report(y_test, y_pred))
print("ROC-AUC:", roc_auc_score(y_test, y_prob))

              precision    recall  f1-score   support

          No       0.89      0.73      0.80       131
         Yes       0.88      0.96      0.91       269

    accuracy                           0.88       400
   macro avg       0.88      0.84      0.86       400
weighted avg       0.88      0.88      0.88       400

ROC-AUC: 0.9001674281336021


In [39]:
import pandas as pd

importance = pd.Series(model.feature_importances_, index=X.columns)
importance = importance.sort_values(ascending=False)

print(importance.head(10))

User_Total_Transactions              0.242299
User_Count_Transactions_In_States    0.235146
User_Unique_Merchants_Count          0.173508
User_Unique_MCC_Count                0.148693
User_Unique_City_Count               0.117107
User_Std_Day_of_Month                0.083247
User_Frequency_by_Month              0.000000
dtype: float64
